In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import seaborn as sns 
from pandas.api.types import CategoricalDtype
import warnings
warnings.filterwarnings("ignore")

# STAT390 Legal Aid Project, Sarah Abara
*Cleaning and Processing Data for Final Tableau Dashboard*

In [2]:
# List of (month label, file path) tuples
month_files = [
    ('April 2024', 'All Calls by Month/April 2024.xlsx'),
    ('August 2024', 'All Calls by Month/August 2024.xlsx'),
    ('December 2024', 'All Calls by Month/December 2024.xlsx'),
    ('February 2025', 'All Calls by Month/February 2025.xlsx'),
    ('January 2025', 'All Calls by Month/January 2025.xlsx'),
    ('July 2024', 'All Calls by Month/July 2024.xlsx'),
    ('June 2024', 'All Calls by Month/June 2024.xlsx'),
    ('March 2025', 'All Calls by Month/March 2025.xlsx'),
    ('May 2024', 'All Calls by Month/May 2024.xlsx'),
    ('October 2024', 'All Calls by Month/October 2024.xlsx'),
    ('November 2024', 'All Calls by Month/November 2024.xlsx'),
    ('September 2024', 'All Calls by Month/September 2024.xlsx'),
]

# Read and combine all DataFrames
dfs = []

for month, path in month_files:
    df = pd.read_excel(path)
    df['Month'] = month

    # moving month column to first column
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('Month')))
    df = df[cols]
    
    dfs.append(df)

# Concatenate all into a single DataFrame
all_calls = pd.concat(dfs, ignore_index = True)

In [3]:
# Special Intake Line Dictionary
intake_line_numbers = {
    "A2J Immigration": 13123478347,
    "A2J Immigration Toll Free": 18882652188,
    "Austin Intake VM": 13124235904,
    "Bankruptcy Helpdesk VM": 13122296344, 
    "CLASP VM": 13124235900,
    "Criminal Records": 13122296071,
    "Education Law Referrals VM": 13123478392,
    "Fair Housing Intake VM": 13124235909, 
    "HIV Intake VM": 13123478309,
    "JEHD": 13122296072,
    "Legal Clinics": 13124235938,
    "OP Appeals Project": 13124312101, 
    "Veterans Rights Project VM": 13123478340,
    "Trafficking Survivors Assistance Project": 13122296073,
    "Nursing Home Ombudsman": 13122296079,
    "Markham Eviction Help Desk": 13122296014,
    "Farmworker": 13124312299
    
}

# CHECK IN WITH KRISH: is "Migrant Legal Assistance Program": 13124312299 a special intake line or a main number line?
    # because as of May 20, this number was called: Farmworker main number calls (312-431-2299)
    # FINAL ANSWER: document farmworker as a special intake line

In [4]:
# Main Line Number Dictionary
main_line_numbers = {
    
    "Legal Aid main number calls": 13123411070,
    "Transfers to the English main menu": 13125068646,
    "Transfers to the Spanish main menu": 13125068647,
    "Old ADAPT number calls that route to the Legal Aid main menu": 13122296080,
    "Old Benefits Enrollment number calls that route to the Legal Aid main menu": 13123478342
}

In [5]:
# # Creating a set from the previously defined dictionaries for the purpose of quick lookup
special_intake_numbers = set(intake_line_numbers.values())
main_numbers = set(main_line_numbers.values())

# Creating a mapping from number to name for special intake lines and main lines
number_to_special_intake_name = {v: k for k, v in intake_line_numbers.items()}
number_to_main_line_name = {v: k for k, v in main_line_numbers.items()}

# Creating Relevant columns with names of each line based on previous mapping and the dictionary 
all_calls['Special Intake Line Name'] = all_calls['Called number'].map(number_to_special_intake_name)
all_calls['Main Number Name'] = all_calls['Called number'].map(number_to_main_line_name)

# Combine both dictionaries into one
number_to_line_name = {**number_to_special_intake_name, **number_to_main_line_name}

# Create the unified column
all_calls['Line Name'] = all_calls['Called number'].map(number_to_line_name)

# Classifying Line Type
all_calls['Line Type'] = all_calls['Called number'].apply(
    lambda x: (
        "Unknown" if pd.isna(x) else
        "Main Number" if x == main_numbers else
        "Special Intake Lines" if x in special_intake_numbers else
        "Non-Special Intake Lines"
    )
)

In [6]:
# Adding New Columns to clearly visualize on Tableau

# Convert datetime columns
datetime_columns = ['Start time', 'Answer time', 'Release time', 'Report time']

for col in datetime_columns:
    if col in all_calls.columns:
        all_calls[col] = pd.to_datetime(all_calls[col], errors='coerce')

all_calls['Hour'] = all_calls['Start time'].dt.hour
    # what hour did they call based on start time

all_calls['DayOfWeek'] = all_calls['Start time'].dt.day_name()
    # what day of the week did they call based on start time

all_calls['Day_Type'] = np.where(all_calls['DayOfWeek'].isin(['Saturday', 'Sunday']), 'Weekend', 'Weekday')
    # is it a weekend or a weekday 

all_calls['Month_Day'] = all_calls['Start time'].dt.strftime('%B %-d')
    # need the day of the month and the month itself

all_calls['Month_Year'] = all_calls['Start time'].dt.strftime('%B %Y')
    # need the month and the year

all_calls['Business hours'] =  all_calls['Start time'].apply(
    lambda t: 'Business Hours' if pd.to_datetime('08:00:00').time() <= t.time() <= pd.to_datetime('17:00:00').time()
    else 'Outside Business Hours'
)
    # business hours vs non-business hours

hour_labels = [
    f"{(h % 12 or 12)}:00 {'AM' if h < 12 else 'PM'} - {(h % 12 or 12)}:59 {'AM' if h < 12 else 'PM'}"
    for h in range(24)
]
    # creating hour labels for time buckets

all_calls['Time bucket'] = all_calls['Hour'].apply(
    lambda h: f"{(h % 12 or 12)}:00 {'AM' if h < 12 else 'PM'} - {(h % 12 or 12)}:59 {'AM' if h < 12 else 'PM'}"
)
    # adding the time bucket labels

time_bucket_type = CategoricalDtype(categories=hour_labels, ordered=True)
all_calls['Time bucket'] = all_calls['Time bucket'].astype(time_bucket_type)
    # Setting as ordered categorical for Tableau-friendly sorting

In [7]:
all_calls['Duration'] = pd.to_numeric(all_calls['Duration'], errors='coerce')
    # making sure duration is in seconds and is numeric

all_calls['Duration_Seconds'] = all_calls['Duration']
all_calls['Duration_Minutes'] = all_calls['Duration'] / 60
    # putting duration in seconds and minutes

all_calls['Date'] = all_calls['Start time'].dt.date
    # need the date

start_date = pd.to_datetime('2024-04-07', utc = True)
end_date = pd.to_datetime('2025-03-15', utc = True)

all_calls = all_calls[(all_calls['Start time'] >= start_date) & (all_calls['Start time'] <= end_date)]

In [39]:
# -- Call Leg Tracking --
# Creating a leg group key using only stable identifiers ---
leg_key_cols = [
    'Correlation ID',
    'Date',
    'Start time',
    'Called number',
    'Special Intake Line Name',
    'PSTN vendor name'
]
all_calls['Leg Group Key'] = all_calls[leg_key_cols].astype(str).agg('-'.join, axis=1)

# Assigning leg numbers based on unique leg groups 
journey_groups = all_calls.groupby(['Correlation ID', 'Date'])

all_calls['Leg Number'] = journey_groups['Leg Group Key'].transform(lambda x: pd.factorize(x)[0] + 1)
all_calls['Leg Number'] = all_calls['Leg Number'].astype(int)
all_calls['Total Legs'] = journey_groups['Leg Group Key'].transform('nunique')

For all inbound calls: To classify Direct vs Transfer, we need to look at two main aspects of the call:

1. Call Type — Is it an "Inbound" call?

2. Call Flow — Did it go directly to a special intake line, or did it first hit a main menu number and then transfer?

In [ ]:
# === Step 1: Sorting and tagging first leg ===
all_calls = all_calls.sort_values(by=['Correlation ID', 'Start time']).copy()
all_calls['Is First Leg'] = all_calls.groupby('Correlation ID').cumcount() == 0

# === Step 2: Assigning Call Type ONLY to first legs ===
def assign_call_type(row):
    if row['Is First Leg']:
        if pd.isna(row['PSTN vendor name']):
            return 'Internal'
        elif row['PSTN vendor name'] == 'CallTower' and row['Direction'] == 'TERMINATING':
            return 'Inbound'
        elif row['PSTN vendor name'] == 'CallTower' and row['Direction'] == 'ORIGINATING':
            return 'Outbound'
        else:
            return 'Unknown'
    else:
        return None  # To not assign to non-first legs

all_calls['Call Type'] = all_calls.apply(assign_call_type, axis=1)

# First, flagging for Correlation IDs with a transfer leg not in the first leg)
def has_transfer_leg(sub_df):
    first_leg_index = sub_df['Is First Leg'].idxmax()
    return sub_df.loc[sub_df.index != first_leg_index, 'PSTN vendor name'].isna().any()

transfer_flags = all_calls.groupby('Correlation ID').apply(has_transfer_leg).rename("Has Transfer").reset_index()
all_calls = all_calls.merge(transfer_flags, on='Correlation ID', how='left')

# Now assigning Direct/Transfer value using Call Type from the first leg and transfer flag
def assign_direct_transfer(row):
    if row['Has Transfer']:
        return 'Transfer'
    elif row['Call Type'] in ['Inbound', 'Outbound']:
        return 'Direct'
    else:
        return None

all_calls['Direct_Transfer'] = all_calls.apply(assign_direct_transfer, axis=1)

In [ ]:
# Creating a Unified Line Category Column for easier visualization in Tableau 

def categorize_line(number):
    if pd.isna(number):
        return "Unknown"
    elif number in special_intake_numbers:
        return "Intake Line"
    elif number in main_numbers:
        return "Main Menu Number"
    else:
        return "Other Line"

all_calls['Line Category'] = all_calls['Called number'].apply(categorize_line)

In [ ]:
print(all_calls.columns)
print(all_calls.shape)

In [ ]:
columns_to_keep = ['Month','Correlation ID', 'Direction', 'Duration_Seconds', 'Duration_Minutes',
                   'Duration', 'Called number', 'Hour','DayOfWeek', 'Date',
                   'PSTN vendor Org ID', 'PSTN vendor name', 'Line Category',
                   'Month', 'Start time', 'Answer time', 'Call type',
                   'Call outcome','Special Intake Line Name', 'Main Number Name', 'Line Type', 'Hour',
                   'DayOfWeek', 'Day_Type', 'Month_Day', 'Month_Year', 'Business hours',
                   'Time bucket', 'Call_Type', 'Duration_Seconds', 'Duration_Minutes',
                   'Date', 'Inbound_Type', 'Is_Direct_Inbound', 'Is_Transfer_Inbound',
                   'Leg Group Key', 'Leg Number', 'Total Legs', 'Line Name'
                   ]

all_calls_filtered = all_calls[columns_to_keep]

df = pd.DataFrame(all_calls_filtered)

df.to_csv('all_calls_filtered_final.csv', index=False)